# 01 — Baseline EfficientNetB0

Ce notebook entraîne un classifieur à 27 classes par **transfer learning** :

1. apprentissage de la tête de classification avec le backbone ImageNet gelé ;
2. fine-tuning des 30 dernières couches avec un learning rate faible ;
3. sauvegarde automatique du meilleur modèle selon la loss de validation.

Aucun CLAHE n'est appliqué : ce run constitue la référence expérimentale. EfficientNet Keras inclut
déjà sa couche de rescaling et attend des pixels flottants dans l'intervalle `[0, 255]`.


In [12]:
from pathlib import Path
import json
import os
import random

import certifi
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Python installé depuis python.org sur macOS ne référence pas toujours le trousseau
# système. On fournit à urllib/Keras un bundle CA explicite sans désactiver SSL.
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

def find_project_root() -> Path:
    """Permet d'exécuter le notebook depuis sa racine ou son propre dossier."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "raw_x.csv").exists() and (candidate / "images").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable (data/raw_x.csv et images/ attendus).")

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
IMAGE_DIR = ROOT / "images" / "image_train"
ARTIFACTS_DIR = ROOT / "artifacts" / "image_classification"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("KERAS_HOME", str(ARTIFACTS_DIR / ".keras"))

print("Racine :", ROOT)
print("Artefacts :", ARTIFACTS_DIR)
print("Certificats SSL :", os.environ["SSL_CERT_FILE"])


Racine : /Users/wolfidy7/projects/sep25_alt2_iia_rakuten_2
Artefacts : /Users/wolfidy7/projects/sep25_alt2_iia_rakuten_2/artifacts/image_classification
Certificats SSL : /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/certifi/cacert.pem


In [13]:
import tensorflow as tf

tf.keras.utils.set_random_seed(SEED)

TARGET_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

with open(ARTIFACTS_DIR / "class_mapping.json", encoding="utf-8") as handle:
    class_mapping = json.load(handle)

index_to_code = [int(value) for value in class_mapping["index_to_code"]]
code_to_index = {code: index for index, code in enumerate(index_to_code)}
NUM_CLASSES = len(index_to_code)

manifest = pd.read_csv(ARTIFACTS_DIR / "image_manifest.csv")
manifest["class_index"] = manifest["prdtypecode"].map(code_to_index)
assert manifest["class_index"].notna().all()

def decode_resize(path, label, weight):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize_with_pad(image, *TARGET_SIZE, antialias=True)
    image = tf.cast(image, tf.float32)  # EfficientNet attend des pixels dans [0, 255].
    return image, tf.cast(label, tf.int32), tf.cast(weight, tf.float32)

def make_dataset(frame, training=False, transform=None, weighted=False):
    paths = frame["image_path"].astype(str).to_numpy()
    labels = frame["class_index"].astype("int32").to_numpy()
    weights = frame["sample_weight"].astype("float32").to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, weights))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_resize, num_parallel_calls=AUTOTUNE)
    if transform is not None:
        ds = ds.map(transform, num_parallel_calls=AUTOTUNE)
    if not weighted:
        ds = ds.map(lambda x, y, w: (x, y), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_df = manifest.query("split == 'train'").copy()
val_df = manifest.query("split == 'validation'").copy()
print(f"Train={len(train_df):,} | Validation={len(val_df):,} | Classes={NUM_CLASSES}")


Train=65,900 | Validation=16,475 | Classes=27


## Pipeline et architecture

Le redimensionnement conserve le ratio et ajoute du padding. L'augmentation reste modérée ; le flip
horizontal est volontairement absent car beaucoup d'images contiennent du texte, des jaquettes ou des logos.


In [14]:
from tensorflow import keras
from tensorflow.keras import layers

def build_model():
    augmentation = keras.Sequential(
        [
            layers.RandomRotation(0.03, fill_mode="reflect", seed=SEED),
            layers.RandomZoom(0.10, fill_mode="reflect", seed=SEED),
            layers.RandomContrast(0.10, seed=SEED),
        ],
        name="augmentation",
    )
    inputs = keras.Input(shape=(*TARGET_SIZE, 3), name="image")
    x = augmentation(inputs)
    local_weights = ARTIFACTS_DIR / ".keras" / "models" / "efficientnetb0_notop.h5"
    weights = str(local_weights) if local_weights.exists() else "imagenet"
    print("Poids EfficientNet :", weights)
    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        weights=weights,
        input_shape=(*TARGET_SIZE, 3),
    )
    backbone.trainable = False
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.Dropout(0.30, seed=SEED)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="prdtypecode")(x)
    return keras.Model(inputs, outputs), backbone

def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy"),
        ],
    )

def callbacks_for(run_dir):
    run_dir.mkdir(parents=True, exist_ok=True)
    return [
        keras.callbacks.ModelCheckpoint(
            run_dir / "best_model.keras", monitor="val_loss", save_best_only=True
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7
        ),
        keras.callbacks.CSVLogger(run_dir / "training_log.csv", append=True),
    ]

def unfreeze_last_layers(backbone, count=30):
    backbone.trainable = True
    for layer in backbone.layers[:-count]:
        layer.trainable = False
    # Les statistiques de BatchNorm restent figées sur ce dataset relativement petit.
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False


In [15]:
RUN_NAME = "efficientnet_baseline"
TRAIN_TRANSFORM = None
VAL_TRANSFORM = None


In [16]:
RUN_DIR = ARTIFACTS_DIR / RUN_NAME
train_ds = make_dataset(train_df, training=True, transform=TRAIN_TRANSFORM, weighted=True)
val_ds = make_dataset(val_df, training=False, transform=VAL_TRANSFORM, weighted=False)
run_callbacks = callbacks_for(RUN_DIR)

model, backbone = build_model()
compile_model(model, learning_rate=1e-3)
model.summary()


Poids EfficientNet : /Users/wolfidy7/projects/sep25_alt2_iia_rakuten_2/artifacts/image_classification/.keras/models/efficientnetb0_notop.h5


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling          │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ prdtypecode (Dense)             │ (None, 27)             │        34,587 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,084,158 (15.58 MB)

 Trainable params: 34,587 (135.11 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

## Phase 1 — Backbone gelé


In [17]:
EPOCHS_FROZEN = 8
history_frozen = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN,
    callbacks=run_callbacks,
)


Epoch 1/8


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


2060/2060 ━━━━━━━━━━━━━━━━━━━━ 582s 280ms/step - accuracy: 0.4791 - loss: 1.8102 - top3_accuracy: 0.7087 - val_accuracy: 0.5422 - val_loss: 1.5384 - val_top3_accuracy: 0.7666 - learning_rate: 0.0010
Epoch 2/8
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 587s 285ms/step - accuracy: 0.5303 - loss: 1.6090 - top3_accuracy: 0.7592 - val_accuracy: 0.5579 - val_loss: 1.4886 - val_top3_accuracy: 0.7791 - learning_rate: 0.0010
Epoch 3/8
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 591s 287ms/step - accuracy: 0.5407 - loss: 1.5708 - top3_accuracy: 0.7667 - val_accuracy: 0.5593 - val_loss: 1.4780 - val_top3_accuracy: 0.7836 - learning_rate: 0.0010
Epoch 4/8
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 574s 279ms/step - accuracy: 0.5458 - loss: 1.5531 - top3_accuracy: 0.7705 - val_accuracy: 0.5616 - val_loss: 1.4745 - val_top3_accuracy: 0.7806 - learning_rate: 0.0010
Epoch 5/8
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 572s 277ms/step - accuracy: 0.5455 - loss: 1.5456 - top3_accuracy: 0.7734 - val_accuracy: 0.5693 - val_loss: 1.4508 - val_top3_accura

## Phase 2 — Fine-tuning partiel


In [18]:
unfreeze_last_layers(backbone, count=30)
compile_model(model, learning_rate=1e-5)

start_epoch = len(history_frozen.history["loss"])
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=start_epoch,
    epochs=start_epoch + 12,
    callbacks=run_callbacks,
)

model.save(RUN_DIR / "final_model.keras")
print("Modèle sauvegardé dans", RUN_DIR)


Epoch 9/20
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 644s 311ms/step - accuracy: 0.5710 - loss: 1.4444 - top3_accuracy: 0.7937 - val_accuracy: 0.5833 - val_loss: 1.3949 - val_top3_accuracy: 0.7987 - learning_rate: 1.0000e-05
Epoch 10/20
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 649s 315ms/step - accuracy: 0.5820 - loss: 1.4058 - top3_accuracy: 0.8012 - val_accuracy: 0.5846 - val_loss: 1.3815 - val_top3_accuracy: 0.8006 - learning_rate: 1.0000e-05
Epoch 11/20
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 662s 321ms/step - accuracy: 0.5919 - loss: 1.3710 - top3_accuracy: 0.8082 - val_accuracy: 0.5900 - val_loss: 1.3643 - val_top3_accuracy: 0.8039 - learning_rate: 1.0000e-05
Epoch 12/20
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 709s 344ms/step - accuracy: 0.5971 - loss: 1.3501 - top3_accuracy: 0.8131 - val_accuracy: 0.5951 - val_loss: 1.3496 - val_top3_accuracy: 0.8074 - learning_rate: 1.0000e-05
Epoch 13/20
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 698s 339ms/step - accuracy: 0.6044 - loss: 1.3285 - top3_accuracy: 0.8171 - val_accuracy: 0.5969 -

## Contrôle rapide

    Le notebook 03 réalisera la comparaison complète. Cette cellule permet déjà de détecter un run défaillant.


In [19]:
from sklearn.metrics import classification_report, f1_score

best_model = keras.models.load_model(RUN_DIR / "best_model.keras")
probabilities = best_model.predict(val_ds, verbose=1)
y_pred = probabilities.argmax(axis=1)
y_true = val_df["class_index"].to_numpy()

print("F1 weighted :", f1_score(y_true, y_pred, average="weighted"))
print("F1 macro    :", f1_score(y_true, y_pred, average="macro"))
print(classification_report(y_true, y_pred, target_names=[str(x) for x in index_to_code]))


515/515 ━━━━━━━━━━━━━━━━━━━━ 119s 228ms/step
F1 weighted : 0.6048639872235896
F1 macro    : 0.561220002090212
              precision    recall  f1-score   support

          10       0.45      0.63      0.53       591
          40       0.53      0.54      0.54       476
          50       0.49      0.38      0.42       320
          60       0.68      0.66      0.67       164
        1140       0.63      0.64      0.64       510
        1160       0.89      0.89      0.89       768
        1180       0.62      0.30      0.41       148
        1280       0.45      0.42      0.44       897
        1281       0.42      0.21      0.28       396
        1300       0.65      0.71      0.68       940
        1301       0.61      0.52      0.56       150
        1302       0.46      0.36      0.41       488
        1320       0.48      0.47      0.47       636
        1560       0.57      0.59      0.58       996
        1920       0.78      0.80      0.79       853
        1940       0.60  